In [1]:
import torch

from utils.dataset import SpacecraftDataset
from utils.trainer import Trainer
from utils.evaluator import Evaluator
from utils.utility import get_model
from utils.evaluations import intersection_over_union, show_img_with_boxes


In [5]:
train_dataset = SpacecraftDataset("train",base_labels_path="Data\\labels" , base_image_path="Data\\images" ,data_path="D:\\file\\Bowl\\DetectionTrack")
eval_dataset = SpacecraftDataset("val",base_labels_path="Data\\labels" , base_image_path="Data\\images" ,data_path="D:\\file\\Bowl\\DetectionTrack")
test_set = SpacecraftDataset("test",base_labels_path="Data\\labels" , base_image_path="Data\\images" ,data_path="D:\\file\\Bowl\\DetectionTrack")
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
eval_dataloader = torch.utils.data.DataLoader(eval_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

FileNotFoundError: [WinError 3] Impossibile trovare il percorso specificato: '/home/fabri/PoseBowl/SpacecraftDetection/Data\\labels/train'

In [ ]:
model = get_model()
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = model.to(device)

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

In [ ]:
trainer = Trainer(model, optimizer, lr_scheduler, device)
trainer.train(train_dataloader, eval_dataloader, epochs=1)

In [ ]:
evaluator = Evaluator(model, device)
evaluator.evaluate(test_set)

In [ ]:
model.eval()


data = test_set.get_random_samples(1)

for img, target in data:
    ris = model([img.to(device)])[0]

    if len(ris['scores']) > 0:
        max_id = torch.argmax(ris['scores'])
        best_box = ris['boxes'][max_id].detach().cpu().numpy()
        
        print(intersection_over_union(best_box,target["boxes"][0]).item())
        show_img_with_boxes(img, best_box)